# AI-Powered HR Assistant

This project builds a question-answering assistant for the Nestle HR policy document. It extracts text from the PDF, divides it into overlapping chunks, creates embeddings with OpenAI, and stores them in a persistent Chroma vector database. For each question, the system retrieves relevant passages and provides them as context to GPT-4o, which is instructed to answer from the document and say when information is unavailable.

A Gradio interface presents the query input and generated answer side by side, making it easy to ask questions about the policy.

In [ ]:
import os
import json
import httpx2
import gradio as gr
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

http_client = httpx2.Client(headers={"Accept-Encoding": "gzip, deflate"})

In [ ]:
# Load config from JSON
with open("./config.json", "r") as f:
    config = json.load(f)
# Extract OpenAI API key
openai_api_key = config["openai"]["api_key"]
# Set as environment variable (recommended)
os.environ["OPENAI_API_KEY"] = openai_api_key # key can be hard coded here .
model = ChatOpenAI(model="gpt-4o", http_client=http_client)

In [ ]:
## Load the documents
pdf_files = [
    "./the_nestle_hr_policy_pdf_2012.pdf"
    #"./Insurance/Terms and Conditions for Private Car_2.pdf",
    # "./Books/deep-learning-material-dept-ece-amr-blr-1.pdf"
]

documents = []

for file in pdf_files:
    loader = PyPDFLoader(file)
    docs = loader.load()
    documents.extend(docs)

print("Loaded pages:", len(documents))


In [ ]:
## Split the docs into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)
print("Chunks:", len(chunks))

In [ ]:
## Load the chunks into vector DB
# 1. Initialize the embedding model (Make sure OPENAI_API_KEY is set in your environment variables)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", http_client=http_client)
# 2. Create and persist the Chroma vector database
vector_db = Chroma(
    collection_name="my_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db",  # Where to save data locally, remove if not necessary
)

print("Vectob DB created")

vector_db.add_documents(chunks)

In [ ]:
## Retrieve relevant documents based on a query
retriever = vector_db.as_retriever(search_kwargs={"k":3})

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# prompt chain
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Use only the context below to answer.
Context:
{context}

Question:
{question}

If answer is not found, say "Not available in documents".
""")


In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
)

In [ ]:
query = "Give employment and working conditions at Nestle ?"

answer = rag_chain.invoke(query)

print(answer.content)

In [ ]:
## Side-by-side HR policy assistant

def query_rag_pipeline(message: str) -> str:
    answer = rag_chain.invoke(message)
    return answer.content


with gr.Blocks(title="Nestle HR Policy Assistant") as demo:
    gr.Markdown("# Nestle HR Policy Assistant")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Query")
            query_box = gr.Textbox(
                label="Ask about the HR policy",
                placeholder="Enter your question...",
                lines=8,
            )
            ask_button = gr.Button("Ask", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### Answer")
            answer_box = gr.Markdown("Your answer will appear here.")

    ask_button.click(
        fn=query_rag_pipeline,
        inputs=query_box,
        outputs=answer_box,
    )
    query_box.submit(
        fn=query_rag_pipeline,
        inputs=query_box,
        outputs=answer_box,
    )

demo.launch()